In [1]:
!pip install -q transformers datasets evaluate rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)

In [3]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

train_data = dataset["train"].select(range(1300))   # 🔥 small
val_data = dataset["validation"].select(range(100))

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [4]:
from transformers import BartTokenizer, BartForConditionalGeneration

model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [5]:
model.gradient_checkpointing_enable()

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

torch.cuda.empty_cache()

In [6]:
def preprocess(example):
    model_inputs = tokenizer(
        example["article"],
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        example["highlights"],
        max_length=32,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_data = train_data.map(preprocess)
val_data = val_data.map(preprocess)

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [7]:

rouge = evaluate.load("rouge")


In [8]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Sometimes predictions come as tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # If still logits → convert
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return result

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    fp16=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [12]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,4.262081,0.525494,0.269485,0.495602,0.516587
2,No log,4.400624,0.522822,0.264153,0.490407,0.512784


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=326, training_loss=13.770513803680982, metrics={'train_runtime': 787.6286, 'train_samples_per_second': 3.301, 'train_steps_per_second': 0.414, 'total_flos': 704308995686400.0, 'train_loss': 13.770513803680982, 'epoch': 2.0})

In [13]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 4.4006242752075195, 'eval_rouge1': 0.5228221266182675, 'eval_rouge2': 0.2641531872969379, 'eval_rougeL': 0.4904073683531193, 'eval_rougeLsum': 0.5127840687399963, 'eval_runtime': 10.2431, 'eval_samples_per_second': 9.763, 'eval_steps_per_second': 4.881, 'epoch': 2.0}


In [14]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [15]:
def summarize(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=55,
        min_length=15,              # 🔥 smaller
        num_beams=5,
        length_penalty=2.5,         # 🔥 less aggressive
        no_repeat_ngram_size=3,
        repetition_penalty=1.8,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Test
text = """
India secured a historic victory against Australia in the final match.
The team displayed exceptional batting and bowling performance.
Several players contributed significantly, leading to a dominant win.
"""

print(summarize(text))

India secured a historic victory against Australia in the final match .
The team displayed exceptional batting and bowling performance.


In [16]:
text = """
India achieved a remarkable victory against Australia in the final match of the series, showcasing exceptional performance across all departments. 
The batting lineup delivered a strong start, with key players forming crucial partnerships that built a solid foundation. 
The bowlers maintained consistent pressure, taking important wickets at critical moments and restricting the opposition's scoring opportunities. 
Fielding efforts were equally impressive, with sharp catches and quick ground coverage preventing easy runs. 
Several players stood out for their individual contributions, demonstrating both skill and teamwork throughout the match. 
This victory marks a significant milestone for the team and boosts their confidence ahead of upcoming international tournaments. 
The win also highlights the effectiveness of the team's strategy and preparation, reflecting the hard work put in by both players and coaching staff.
"""

print(summarize(text))

India beat Australia by eight wickets in the final match of the series .
The victory marks a significant milestone for the team and boosts their confidence ahead of upcoming international tournaments .
Several players stood out for their individual contributions


In [17]:
text = """
Artificial Intelligence has rapidly transformed various industries, including healthcare, finance, and transportation. 
Machine learning models are now capable of analyzing vast amounts of data to make accurate predictions and assist in decision-making processes. 
In healthcare, AI systems help doctors diagnose diseases earlier and recommend personalized treatments. 
In finance, algorithms detect fraudulent transactions and optimize investment strategies. 
Despite its advantages, AI also raises concerns related to job displacement, data privacy, and ethical decision-making. 
Researchers and policymakers are actively working to ensure that AI development remains responsible and beneficial for society.
"""

print(summarize(text))

Artificial Intelligence has rapidly transformed various industries, including healthcare, finance, and transportation .
Despite its advantages, AI raises concerns related to job displacement, data privacy, and ethical


In [18]:
text = """
A new startup has emerged in the electric vehicle market, aiming to revolutionize urban transportation with affordable and sustainable solutions. 
The company has developed innovative battery technology that allows faster charging and longer driving ranges. 
Investors have shown strong interest, leading to significant funding in early rounds. 
However, the startup faces challenges such as intense competition, regulatory hurdles, and the need for large-scale infrastructure. 
Experts believe that if executed well, the company could play a key role in accelerating the adoption of electric vehicles globally.
"""

print(summarize(text))

The company has developed innovative battery technology that allows faster charging and longer driving ranges .
Investors have shown strong interest, leading to significant funding in early rounds .
The startup faces challenges such as intense competition
